In [56]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, precision_recall_curve, f1_score,  precision_score, recall_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LeakyReLU, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
from tensorflow.keras.regularizers import l2
import tensorflow.keras.backend as K
import pickle
import os


In [57]:
# ---------------------------
# 1) Cargar datos
# ---------------------------
CSV_PATH = "../data/stroke_dataset.csv"   # <- Ruta corregida al archivo en data/
df = pd.read_csv(CSV_PATH)
df = df.dropna()

target = "stroke"
y = df[target]
X = df.drop(columns=[target])

In [58]:
# Codificar variables categóricas
for col in X.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])


In [59]:
# Dividir en train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [60]:
# Escalar datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [61]:
# Métrica personalizada F1

def f1_metric(y_true, y_pred):
    y_pred = K.round(K.clip(y_pred, 0, 1))  # Asegura que esté entre 0 y 1
    y_true = K.cast(y_true, 'float32')
    y_pred = K.cast(y_pred, 'float32')

    tp = K.sum(y_true * y_pred)
    fp = K.sum((1 - y_true) * y_pred)
    fn = K.sum(y_true * (1 - y_pred))

    precision = tp / (tp + fp + K.epsilon())
    recall = tp / (tp + fn + K.epsilon())
    f1 = 2 * precision * recall / (precision + recall + K.epsilon())

    return f1



In [62]:
# =========================
# 2️⃣ Calcular class_weight (para datos desequilibrados)
# =========================
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = {
    cls: weight for cls, weight in zip(np.unique(y_train), class_weights)
}
print("🔢 Class weights:", class_weight_dict)


🔢 Class weights: {0: 0.526148969889065, 1: 10.06060606060606}


In [63]:
# =========================
# 3️⃣ Definir modelo MLP
# =========================
model = Sequential([
    Dense(128, kernel_regularizer=l2(0.001), input_shape=(X_train.shape[1],)),
    LeakyReLU(alpha=0.1),
    BatchNormalization(),
    Dropout(0.4),

    Dense(64, kernel_regularizer=l2(0.001)),
    LeakyReLU(alpha=0.1),
    BatchNormalization(),
    Dropout(0.3),

    Dense(1, activation='sigmoid')
])



In [64]:
# =========================
# 4️⃣ Compilar modelo
# =========================
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [65]:
# =========================
# 5️⃣ Entrenar modelo
# =========================
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(X_train_scaled, y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=100,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1,
)

Epoch 1/100
125/125 [==============================] - 1s 5ms/step - loss: 0.8095 - accuracy: 0.5846 - val_loss: 0.5714 - val_accuracy: 0.8094
Epoch 2/100
125/125 [==============================] - 0s 3ms/step - loss: 0.7134 - accuracy: 0.6624 - val_loss: 0.5546 - val_accuracy: 0.7844
Epoch 3/100
125/125 [==============================] - 0s 3ms/step - loss: 0.6909 - accuracy: 0.6797 - val_loss: 0.6054 - val_accuracy: 0.7513
Epoch 4/100
125/125 [==============================] - 0s 3ms/step - loss: 0.6409 - accuracy: 0.7058 - val_loss: 0.5972 - val_accuracy: 0.7653
Epoch 5/100
125/125 [==============================] - 0s 3ms/step - loss: 0.6482 - accuracy: 0.7229 - val_loss: 0.6314 - val_accuracy: 0.7262
Epoch 6/100
125/125 [==============================] - 0s 3ms/step - loss: 0.6492 - accuracy: 0.6908 - val_loss: 0.6131 - val_accuracy: 0.7402
Epoch 7/100
125/125 [==============================] - 0s 3ms/step - loss: 0.6404 - accuracy: 0.7131 - val_loss: 0.6418 - val_accuracy: 0.7172

In [66]:
# =========================
# 6️⃣ Evaluación
# =========================
y_pred_prob = model.predict(X_test_scaled).ravel()

# ROC-AUC y PR-AUC
roc_auc = roc_auc_score(y_test, y_pred_prob)
pr_auc = average_precision_score(y_test, y_pred_prob)
print(f"\nROC-AUC: {roc_auc:.3f}")
print(f"PR-AUC: {pr_auc:.3f}")


32/32 [==============================] - 0s 1ms/step

ROC-AUC: 0.826
PR-AUC: 0.179


In [67]:
# =========================
# 🔍 Buscar umbral óptimo (máxima precisión)
# =========================

thresholds = np.linspace(0.1, 0.9, 100)
precisions = []
recalls = []
f1_scores = []

for t in thresholds:
    y_pred_temp = (y_pred_prob >= t).astype(int)
    p = precision_score(y_test, y_pred_temp, zero_division=0)
    r = recall_score(y_test, y_pred_temp, zero_division=0)
    f1 = f1_score(y_test, y_pred_temp, zero_division=0)
    precisions.append(p)
    recalls.append(r)
    f1_scores.append(f1)

best_precision = max(precisions)
best_threshold_precision = thresholds[np.argmax(precisions)]

best_f1 = max(f1_scores)
best_threshold_f1 = thresholds[np.argmax(f1_scores)]

# Using the threshold that maximizes precision for the final evaluation metrics
y_pred_opt = (y_pred_prob >= best_threshold_precision).astype(int)

print(f"📈 Precisión máxima: {best_precision:.3f}")
print(f"🎯 Umbral que maximiza precisión: {best_threshold_precision:.3f}")
print(f"📉 Recall en ese umbral: {recall_score(y_test, y_pred_opt, zero_division=0):.3f}")
print(f"⚖️ F1-score en ese umbral: {f1_score(y_test, y_pred_opt, zero_division=0):.3f}")

print(f"\n⚖️ F1-score máximo: {best_f1:.3f}")
print(f"🎯 Umbral que maximiza F1-score: {best_threshold_f1:.3f}")

📈 Precisión máxima: 0.243
🎯 Umbral que maximiza precisión: 0.779
📉 Recall en ese umbral: 0.180
⚖️ F1-score en ese umbral: 0.207

⚖️ F1-score máximo: 0.251
🎯 Umbral que maximiza F1-score: 0.536


In [68]:
# =========================
# 9️⃣ Visualización
# =========================

# Configurar el backend de matplotlib explícitamente para entornos sin GUI
import matplotlib
matplotlib.use('Agg')  # Backend sin GUI para entornos containerizados
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Evolución de la precisión')
plt.xlabel('Épocas')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Evolución de la pérdida')
plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.legend()
plt.show()

precisions, recalls, thresholds_pr = precision_recall_curve(y_test, y_pred_prob)
plt.figure(figsize=(8, 5))
plt.plot(thresholds_pr, precisions[:-1], label='Precisión')
plt.plot(thresholds_pr, recalls[:-1], label='Recall')
plt.xlabel('Umbral')
plt.ylabel('Valor')
plt.title('Precisión vs Recall según umbral')
plt.legend()
plt.grid(True)
plt.show()

# Calculate F1 scores for the thresholds returned by precision_recall_curve
f1_scores_pr = [f1_score(y_test, (y_pred_prob >= t).astype(int), zero_division=0) for t in thresholds_pr[:-1]]

plt.figure(figsize=(8, 5))
plt.plot(thresholds_pr[:-1], f1_scores_pr, label='F1-score', color='green')
plt.axvline(best_threshold_f1, color='green', linestyle='--', label=f'Umbral F1: {best_threshold_f1:.2f}')
plt.xlabel('Umbral')
plt.ylabel('F1-score')
plt.title('F1-score según umbral')
plt.legend()
plt.grid(True)
plt.show()


print("📊 Reporte con umbral de máxima precisión:")
print(classification_report(y_test, (y_pred_prob >= 0.900).astype(int), digits=3))

print("📊 Reporte con umbral de máximo F1-score:")
print(classification_report(y_test, (y_pred_prob >= 0.625).astype(int), digits=3))

/tmp/ipykernel_10639/1039023188.py:17: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/tmp/ipykernel_10639/1039023188.py:26: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/tmp/ipykernel_10639/1039023188.py:37: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


📊 Reporte con umbral de máxima precisión:
              precision    recall  f1-score   support

           0      0.951     0.995     0.972       947
           1      0.167     0.020     0.036        50

    accuracy                          0.946       997
   macro avg      0.559     0.507     0.504       997
weighted avg      0.911     0.946     0.925       997

📊 Reporte con umbral de máximo F1-score:
              precision    recall  f1-score   support

           0      0.966     0.895     0.929       947
           1      0.168     0.400     0.237        50

    accuracy                          0.871       997
   macro avg      0.567     0.648     0.583       997
weighted avg      0.926     0.871     0.895       997



/tmp/ipykernel_10639/1039023188.py:50: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [69]:
# Guardar el modelo
model.save("modelo_stroke.h5")

# Guardar el nombre del archivo o metadatos en un .pkl
import joblib
joblib.dump({"modelo_path": "modelo_stroke.h5"}, "modelo_info.pkl")



['modelo_info.pkl']

In [70]:
# Guardar el modelo como pickle en data/
import pickle

# Guardar modelo como pickle
with open('../data/mlp_model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Guardar el scaler entrenado también
with open('../data/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("✅ Modelo guardado como pickle en '../data/mlp_model.pkl'")
print("✅ Scaler guardado como pickle en '../data/scaler.pkl'")

# Guardar metadatos adicionales
import joblib
metadata = {
    "modelo_path_h5": "modelo_stroke.h5",
    "modelo_path_pkl": "../data/mlp_model.pkl",
    "scaler_path": "../data/scaler.pkl",
    "best_threshold_precision": best_threshold_precision,
    "best_threshold_f1": best_threshold_f1,
    "roc_auc": roc_auc,
    "pr_auc": pr_auc
}
joblib.dump(metadata, "../data/modelo_info.pkl")

print("✅ Metadatos guardados en '../data/modelo_info.pkl'")

INFO:tensorflow:Assets written to: ram://7d91e969-f3f8-4660-87eb-00f6b1d1cdcf/assets
✅ Modelo guardado como pickle en '../data/mlp_model.pkl'
✅ Metadatos guardados en '../data/modelo_info.pkl'
